# Convert ORC to CSV with chDB

In-process ClickHouse engine. Run `./generate.sh` first to create `./data/events.orc`.
Companion article: https://clickhouse.com/resources/engineering/convert-orc-to-csv

In [ ]:
import os
import chdb

os.chdir("data")

Inspect the schema chDB infers from the ORC footer. `tags` is a typed `Map`, which CSV has no native shape for.

In [ ]:
chdb.query("DESCRIBE file('events.orc')", "DataFrame")[["name", "type"]]

Convert ORC to CSV in one statement, flattening the `Map` into scalar columns so the CSV stays tabular. `INTO OUTFILE` writes the file from inside chDB.

In [ ]:
chdb.query("""
SELECT
  event_time,
  event_id,
  country,
  action,
  amount,
  tags['utm_source'] AS utm_source,
  tags['device']     AS device
FROM file('events.orc')
INTO OUTFILE 'events_chdb.csv' TRUNCATE FORMAT CSVWithNames
""")

Read the converted CSV back to confirm it round-trips.

In [ ]:
print(chdb.query(
    "SELECT * FROM file('events_chdb.csv') ORDER BY event_id LIMIT 5",
    "PrettyCompact",
))